# AMT – interactive exploration

This notebook is now a **thin wrapper** around the `amt` package. The heavy lifting lives in `amt/core.py`, `amt/viz.py` and `amt/export.py`.

Install once:

```bash
pip install -e .[notebook]
```


## 1. Load a TTL file

In [ ]:
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from amt import (
    load_amt, do_reasoning, check_consistency,
    show_in_notebook,
    export_ttl, export_cypher,
    local_name,
)

amt = None  # populated by the upload widget
out = widgets.Output()

def _on_upload(change):
    global amt
    with out:
        out.clear_output()
        uploaded = uploader.value
        if not uploaded:
            return
        info = uploaded[0] if isinstance(uploaded, tuple) else list(uploaded.values())[0]
        ttl = info['content'].tobytes().decode('utf-8') if hasattr(info['content'], 'tobytes') else info['content'].decode('utf-8')
        print(f"📂 {info['name']} ({len(ttl):,} chars)")
        amt = load_amt(ttl)
        a = amt
        print(f"✓ {len(a['concepts'])} Concepts | {len(a['roles'])} Roles | "
              f"{len(a['nodes'])} Nodes | {len(a['edges'])} Edges | {len(a['axioms'])} Axioms")

uploader = widgets.FileUpload(accept='.ttl', multiple=False, description='Upload TTL', button_style='primary')
uploader.observe(_on_upload, names='value')
display(widgets.VBox([widgets.HTML('<b>Select an AMT-compatible Turtle (.ttl) file:</b>'), uploader, out]))

# OR load directly from disk (skip the widget):
# amt = load_amt('examples/PotterAttributionExample.ttl')

## 2. Inspect the ontology

In [ ]:
assert amt is not None, '⚠ Upload or load a TTL file first.'

df_concepts = pd.DataFrame([
    {'IRI': local_name(c['iri']), 'Label': c['label'], 'Placeholder': c['placeholder']}
    for c in amt['concepts'].values()
])
df_concepts

## 3. Visualise – assertions only

In [ ]:
show_in_notebook(amt['nodes'], amt['edges'], amt['concepts'])

## 4. Visualise – with reasoning

Inferred edges show as **red dashed arrows**; their weights come from the fuzzy logic operator declared in each axiom.

In [ ]:
reasoned = do_reasoning(amt['edges'], amt['axioms'])
inferred = [e for e in reasoned if e.get('inferred')]
print(f'{len(inferred)} inferred edges')
show_in_notebook(amt['nodes'], amt['edges'], amt['concepts'], reasoning=True, axioms=amt['axioms'])

## 5. Consistency check

In [ ]:
ok, violations = check_consistency(amt['edges'], amt['axioms'])
if ok:
    print('✓ consistent')
else:
    for v in violations:
        print(' ·', v)

## 6. Export

In [ ]:
ttl = export_ttl(amt['nodes'], amt['edges'], amt['concepts'], amt['roles'],
                 amt['axioms'], rdf_graph=amt['graph'], prefix=amt['prefix'],
                 with_reasoning=True)
Path('amt-export.ttl').write_text(ttl, encoding='utf-8')

cypher = export_cypher(amt['nodes'], amt['edges'], amt['axioms'], with_reasoning=True)
Path('amt-export.cypher').write_text(cypher, encoding='utf-8')

print('✓ wrote amt-export.ttl and amt-export.cypher')